In [1]:
%reload_kedro

                    INFO     Resolved project path as:                                              ]8;id=10447;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=59566;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py#175\175]8;;\
                             /Users/nicolasbetancourt/Documents/GitHub/serpientes-de-colombia.                     
                             To set a different path, run '%reload_kedro <project_root>'                           

file
data/02_intermediate/train_labels_and_paths.csv
<class 'str'>
file
data/02_intermediate/validation_labels_and_paths.csv
<class 'str'>
file
data/02_intermediate/test_labels_and_paths.csv
<class 'str'>


                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=425929;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=694396;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro_telemetry/plugin.py#273\273]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Kedro project serpientes_de_colombia                                   ]8;id=640413;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=418479;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py#141\141]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=910477;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=492587;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py#142\142]8;;\
                             'pipelines'                                                                           

                    INFO     Registered line magic 'run_viz'                                        ]8;id=925145;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=264437;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/ipython/__init__.py#148\148]8;;\

In [2]:
import numpy as np
import pandas as pd
from torch.utils.data import random_split


In [3]:
image_urls   =catalog.load('image_urls@pandas')
train_validation_size   =catalog.load('params:train_validation_size') 
validation_size=catalog.load('params:validation_size')

                    INFO     Loading data from image_urls@pandas (CSVDataset)...                ]8;id=329876;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=946013;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

                    INFO     Loading data from params:train_validation_size (MemoryDataset)...  ]8;id=303330;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=45778;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

                    INFO     Loading data from params:validation_size (MemoryDataset)...        ]8;id=454547;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=681373;file:///opt/miniconda3/envs/serpientes_env/lib/python3.11/site-packages/kedro/io/data_catalog.py#539\539]8;;\

In [4]:
image_urls=image_urls[image_urls['label']!='macabrel']

In [8]:
validation_size

0.1

In [9]:
# Number of samples on each class
in_train_fraction=train_validation_size*image_urls['label'].value_counts().min()
# 1. Assign a random number to each row
# 2. Order the rows based on the random number within each label group
# 3. Use the first train_validation_size % (e.g. 80%) as train_validation set
train_validation=image_urls[image_urls.assign(random=np.random.uniform(0,1)).groupby(['label'])['random'].transform('rank','first')<=min_train_fraction]
# Test are the set of all observations outside train or validation
test=image_urls[~image_urls['observation_id'].isin(train_validation['observation_id'])]

# Number of samples on each class
min_validation_fraction=validation_size*train_validation['label'].value_counts().min()
# 1. Assign a random number to each row
# 2. Order the rows based on the random number within each label group
# 3. Use the first train_validation_size % (e.g. 80%) as train_validation set
validation=train_validation[train_validation.assign(random=np.random.uniform(0,1)).groupby(['label'])['random'].transform('rank','first')<=min_validation_fraction]
# Train are the set of all observations outside validation
train=train_validation[~train_validation['observation_id'].isin(validation['observation_id'])]

In [10]:
image_urls['label'].value_counts()



label
vibora          1658
coral           1655
falsa_vibora    1655
falsa_coral     1655
Name: count, dtype: int64

In [11]:
train_validation['label'].value_counts()


label
vibora          1324
coral           1324
falsa_vibora    1324
falsa_coral     1324
Name: count, dtype: int64

In [12]:
test['label'].value_counts()



label
vibora          334
coral           331
falsa_coral     330
falsa_vibora    328
Name: count, dtype: int64

In [13]:
train['label'].value_counts()



label
vibora          1192
coral           1192
falsa_vibora    1192
falsa_coral     1190
Name: count, dtype: int64

In [14]:
validation['label'].value_counts()



label
vibora          132
coral           132
falsa_vibora    132
falsa_coral     132
Name: count, dtype: int64